# 0. Importing all the required libraries

In [117]:
uv pip install scikit_learn

Note: you may need to restart the kernel to use updated packages.


Using Python 3.15.0 environment at: venv
Checked 1 package in 8ms


In [118]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix

# 1. Load the datasets

In [119]:
df = pd.read_csv(r"C:\Users\izzat\OneDrive\Documents\Code\Titanic\datasets\Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 2. Inspect shape and missing values

In [120]:
print("Rows and Columns:", df.shape)
print("\nMissing values per column:")
print(df.isnull().sum())

Rows and Columns: (891, 12)

Missing values per column:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


# 3. Handle missing values & clean columns

In [121]:
# Fill missing Age with median age
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fill missing Embarked with the most common port
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Drop columns that don't help general predictions or have too many missing values
df = df.drop(columns=["Cabin", "Ticket", "Name", "PassengerId"], errors="ignore")

print(df.isnull().sum())

df.head()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


# 5. Convert text to numbers

In [122]:
# Convert Sex (male: 0, female: 1)
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,False,True
1,1,1,1,38.0,1,0,71.2833,False,False
2,1,3,1,26.0,0,0,7.9250,False,True
3,1,1,1,35.0,1,0,53.1000,False,True
4,0,3,0,35.0,0,0,8.0500,False,True


# 6. Separate features and target

In [123]:
df_features = df.drop("Survived", axis=1)
df_target = df["Survived"]

df_target.head()

df_features.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,3,0,22.0,1,0,7.2500,False,True
1,1,1,38.0,1,0,71.2833,False,False
2,3,1,26.0,0,0,7.9250,False,True
3,1,1,35.0,1,0,53.1000,False,True
4,3,0,35.0,0,0,8.0500,False,True


# 7. Machine Learning

In [124]:
X_train, X_test, y_train, y_test = train_test_split(
    df_features, df_target, test_size=0.2, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 712
Testing samples: 179


In [125]:
# Initialize and train
model = RandomForestClassifier(n_estimators=100, random_state=13)
model.fit(X_train, y_train)

# Evaluate predictions
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Model Accuracy: {accuracy:.2%}")
print("\nClassification Report:\n", classification_report(y_test, predictions))

Model Accuracy: 82.12%

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.86      0.85       105
           1       0.79      0.77      0.78        74

    accuracy                           0.82       179
   macro avg       0.82      0.81      0.81       179
weighted avg       0.82      0.82      0.82       179



In [126]:
feature_importance = (
    pd.Series(model.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
)

print("Top Features Influencing Survival:")
print(feature_importance)

Top Features Influencing Survival:
Fare          0.275032
Sex           0.263044
Age           0.257057
Pclass        0.083728
SibSp         0.052210
Parch         0.036704
Embarked_S    0.022254
Embarked_Q    0.009971
dtype: float64


In [127]:
cm = confusion_matrix(y_test, predictions)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[90 15]
 [17 57]]
